# 🧪 POC — Qwen 3.5 4B no Google Colab (Ollama)
## Testando Qwen 3.5 4B via Ollama no Colab gratuito

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/poc_ollama_qwen35_4b_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Objetivo:** Testar se o Qwen 3.5 4B roda bem no Colab T4 via Ollama.

**Modelo:** `qwen3.5:4b` (~2.7 GB) — menor que Gemma 4 E2B, mas pode overthink.

**⚠️ POC experimental** — não é material de aula.

---


## 1. Verificar GPU

⚠️ Vá em `Runtime → Change runtime type` e selecione **T4 GPU**.

In [ ]:
!nvidia-smi

## 2. Instalar Ollama

In [ ]:
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

print("✅ Ollama instalado!")

## 3. Iniciar Servidor + Baixar Modelo

Qwen 3.5 4B: ~2.7 GB (bem menor que Gemma 4 E2B que é ~7.2 GB).

In [ ]:
import subprocess, time, requests, os

subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(1)

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env={**os.environ}
)

print("⏳ Aguardando Ollama iniciar...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print("✅ Ollama rodando!")
            break
    except:
        time.sleep(1)

!ollama pull qwen3.5:4b

## 4. Warm Up + Teste Rápido

⚠️ Primeira inferência demora ~1-2 min (modelo carrega na GPU). Depois é rápido.

In [ ]:
import json, time

MODEL = "qwen3.5:4b"
OLLAMA_URL = "http://localhost:11434/api/chat"

def chat(messages, model=None, stream=False):
    if model is None:
        model = MODEL
    payload = {"model": model, "messages": messages, "stream": stream, "keep_alive": -1}
    response = requests.post(OLLAMA_URL, json=payload, timeout=300)
    if response.status_code != 200:
        raise Exception(f"Erro {response.status_code}: {response.text}")
    return response if stream else response.json()

# Warm up
print("🔥 Warm up...")
start = time.time()
r = chat([{"role": "user", "content": "Oi"}])
print(f"✅ Warm up em {time.time()-start:.1f}s")
print(f"   Resposta: {r['message']['content'][:80]}")

## 5. Chat Básico

In [ ]:
resultado = chat([
    {"role": "system", "content": "Você é um assistente útil. Responda em português."},
    {"role": "user", "content": "Qual é a capital de Minas Gerais?"}
])
print(f"🤖 {resultado['message']['content']}")
print(f"📊 Tokens: prompt={resultado.get('prompt_eval_count', '?')}, completion={resultado.get('eval_count', '?')}")

## 6. Chat com Memória

In [ ]:
historico = [{"role": "system", "content": "Você é um assistente útil. Responda em português."}]

historico.append({"role": "user", "content": "Meu nome é Lucas."})
r1 = chat(historico)
historico.append({"role": "assistant", "content": r1["message"]["content"]})
print(f"🤖: {r1['message']['content']}")

historico.append({"role": "user", "content": "Qual é o meu nome?"})
r2 = chat(historico)
historico.append({"role": "assistant", "content": r2["message"]["content"]})
print(f"🤖: {r2['message']['content']}")

## 7. API Compatível com OpenAI

O endpoint `/v1/chat/completions` funciona como a API da OpenAI.

In [ ]:
OPENAI_URL = "http://localhost:11434/v1/chat/completions"

payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "Responda em português."},
        {"role": "user", "content": "O que é Python em uma frase?"}
    ],
    "temperature": 0.7,
    "max_tokens": 64,
    "keep_alive": -1
}

response = requests.post(OPENAI_URL, json=payload, timeout=300)
data = response.json()
print(f"🤖 {data['choices'][0]['message']['content']}")
print(f"📊 Model: {data['model']}, Usage: {data['usage']}")

## 8. Benchmark

In [ ]:
start = time.time()
resultado = chat([{"role": "user", "content": "Conte uma história curta sobre um robô que aprende a cozinhar."}])
elapsed = time.time() - start

eval_count = resultado.get("eval_count", 0)
eval_duration = resultado.get("eval_duration", 0) / 1e9
speed = eval_count / eval_duration if eval_duration > 0 else 0

print(f"⏱️ Tempo: {elapsed:.1f}s")
print(f"📊 Tokens: {resultado.get('prompt_eval_count', '?')} prompt, {eval_count} completion")
print(f"🚀 Velocidade: {speed:.1f} tokens/s")
print(f"\n📝 {resultado['message']['content'][:200]}...")